# Notebook 1 - Words, Gaps & Serial-Position Reinstatement

**iEEG high-gamma (60-160 Hz) representational analysis of free recall (FR1, ds004789).**

This notebook is the neural core of the project. It starts from the **processed data**
(aligned per-session matrices) - *no raw EDF, no preprocessing here* - and walks through
the behavioral anchor and the neural steps, ending with cross-session reinstatement,
decoding, and an interactive anatomical brain map.

### Design rule
This notebook holds **no computations and no function definitions**. Every block is a
**call into the `lib/` modules**; the notebook only sets parameters, calls, plots, and saves.
All logic lives in:

| Module | Responsibility |
|---|---|
| `lib.config`        | paths (env-var driven), the single subject list |
| `lib.dataio`        | load processed matrices / channels / scores / annot / AAL locations (Step 2) |
| `lib.behavior`      | behavioral SPC (Step 1) |
| `lib.psth`          | PSTH + inter-trial reliability score (Step 3) |
| `lib.supersubject`  | channel selection + super-subject grand matrices (Step 4) |
| `lib.segments`      | word/gap extraction + time-averaging (Step 5) |
| `lib.rsa`           | word RSM, serial-position RDM, second-order RSA (Steps 7, 8, 9) |
| `lib.reinstatement` | cross-condition RDM, time-resolved decay, maintenance (Steps 10, 11) |
| `lib.decoding`      | nearest-neighbor decoding (Step 12) |
| `lib.brainmap`      | AAL labeling + interactive per-subject HTML brain maps (Step 13) |
| `lib.viz`           | all plotting helpers |

### Pipeline
```
 1  behavioral SPC (opening)         ->  validate primacy/recency
 2  load processed data
 3  responsive electrodes (PSTH)     ->  inter-trial reliability score
 4  super-subject + channel sel.     ->  grand matrices (300 words x N), per session
 5  word & gap extraction + time-averaging
 6  population PSTH (top-100)         ->  grand-average response of selected channels
 7  word-identity RSM (300x300)       ->  cross-session consistency (scatter + zoom)
 8  serial-position RSA               ->  12x12 RDM, decoding diagonal (3 variants)
 9  second-order RSA                  ->  relative representation + full 2nd-order matrices
10  words vs gaps                     ->  reinstatement cross-RDM + words x gaps geometry
11  time-resolved + maintenance       ->  per-window RDMs, decay, maintenance
12  nearest-neighbor decoding         ->  accuracy vs chance (8.3%)
13  electrode brain map (interactive) ->  AAL-labeled HTML per subject + legend
```


## Setup

**Why:** centralize paths and the subject list so nothing is copy-pasted. `lib.config`
resolves the data root from the `IEEG_DATA_ROOT` environment variable (or a local default),
so the notebook is portable across machines.

**Input:** none (reads config / environment).

**Output:** `PATHS` (resolved directories), `SUBJECTS` (the canonical subject/session list).

In [ ]:
import sys
sys.path.append('..')            # repo root on path so `lib` is importable

from lib import (config, dataio, behavior, psth, supersubject,
                 segments, rsa, reinstatement, decoding, brainmap, viz)

PATHS    = config.resolve_paths()          # data_root, processed_dir, results_dir, ...
SUBJECTS = config.SUBJECTS                 # canonical list of (subject, session) pairs
RESULTS  = PATHS.results_dir / 'reinstatement'
viz.set_paths(PATHS)                       # lets the population PSTH load the RAW matrices

print(f'Data root : {PATHS.data_root}')
print(f'Subjects  : {len(set(s for s, _ in SUBJECTS))} subjects, {len(SUBJECTS)} sessions')

## Step 1 - Behavioral Serial Position Curve (opening)

**Why:** before any neural analysis, confirm subjects behaved as the memory literature
predicts. The SPC is the most robust effect in recall research - a **U-shaped curve**:
strong **primacy** at the start, **recency** at the end, weakest in the middle. Validating
it here anchors the neural question (*does the brain encode position distinctly?*) in
behavior we trust.

**Input:** BIDS behavioral `_beh.tsv` files (encoding `WORD` + retrieval `REC_WORD`);
intrusions and repetitions are filtered inside the library.

**Output:** per-subject SPC table -> group-average curve with SEM; saved
`FR1_behavioral_summary.csv` and `Subject_Level_SPC.csv`.

In [ ]:
spc = behavior.compute_spc(SUBJECTS, PATHS, save=True)   # -> tidy DataFrame (subject, session, Pos_1..12)

viz.plot_spc_curve(spc, out=RESULTS / 'SPC_curve.png')   # U-shape: primacy + recency
viz.plot_spc_bars (spc, out=RESULTS / 'SPC_bars.png')    # grouped bars with SEM

## Step 2 - Load processed data

**Why:** load the entry-point artifacts once, into memory, so every downstream step reuses
them. This is the boundary that lets the notebook run *without* preprocessing.

**Input (per session, from `dr-processed/`):** `_aligned_matrix_NormRatio.npy`
(lists x channels x time), `_aligned_channels_RAW.pkl`, `correlations_scores_NormRatio.csv`,
`.annot` (word events), and `*_electrode_locations.csv` (AAL labels).

**Output:** `DATA` - a dict keyed by `{sub}_{ses}` holding matrix, channel names, scores,
word labels, and region map.

In [ ]:
DATA = dataio.load_sessions(SUBJECTS, PATHS)   # dict: sub_ses -> {matrix, channels, scores, words, regions}
print(f'Loaded {len(DATA)} sessions.')

## Step 3 - Responsive electrodes (PSTH + inter-trial reliability)

**Why:** electrodes are placed for clinical reasons, so most carry no task signal. We find
the channels that respond **consistently** to word onset via their inter-trial correlation
score (how repeatable the response pattern is across the 25 lists). This score is the quality
metric that drives channel selection in Step 4. PSTHs are computed linear-mean -> dB to avoid
Jensen-inequality bias.

**Input:** `DATA` (NormRatio matrices + word onsets).

**Output:** per-channel reliability scores (consumed from `correlations_scores_NormRatio.csv`;
pass `recompute=True` to rebuild from the matrices) and per-channel PSTH figures.

In [ ]:
scores = psth.inter_trial_scores(DATA, recompute=False)   # consume artifact; recompute=True rebuilds it

# Inspect a few example responsive channels (linear-mean -> dB, +/- SEM)
viz.plot_psth(DATA, scores, top_k=5, out=RESULTS / 'psth_examples.png')

## Step 4 - Super-subject construction + channel selection

**Why:** single subjects have sparse, idiosyncratic coverage. Pooling all subjects into one
MNI **super-subject** yields a broad, reliable sample of each region. Channels are kept by
four independent criteria - **inter-trial score**, **AAL region**, and the **word/gap time
windows** - and only channels present in **both sessions** survive, so the cross-session
comparison is fair (same feature columns on both days).

**Input:** `DATA`, `scores`; parameters `region` ('all' or an AAL ROI), `top_n`, `min_score`.
Per-channel z-scoring (over the whole session) is applied inside the library.

**Output:** `SEL` - channel-selected, z-scored per-session matrices (lists x selected_ch x time)
+ `df_features` mapping each column to its subject/channel.

In [ ]:
SEL, df_features = supersubject.select_channels(
    DATA, scores,
    region    = 'all',     # or 'IFG' / 'PFC' / 'MTL' / 'Temporal' / 'Occipital' / 'Motor'
    top_n     = 180,
    min_score = -1,        # -1 disables the threshold
    zscore    = True,
)
print(f'Selected sessions: {len(SEL)} | total channels: {df_features.shape[0]}')

## Step 5 - Word & Gap extraction + time-averaging

**Why:** collapse each segment to one response value per channel per position - the unit RSA
operates on. Words and gaps are extracted separately so Step 10 can compare encoding vs the
silent interval. The last gap is only 0.15 s (NaN-padded) per the protocol.

**Input:** `SEL` + `.annot`; `word_window` (<=1.6 s) and `gap_window` (<=0.5 s).

**Output:** four super-subject **grand matrices** - `grand.words_s0/s1` and `grand.gaps_s0/s1`,
each (300 words x N channels) - built by concatenating subjects column-wise.

In [ ]:
segs  = segments.extract_words_and_gaps(
    SEL, PATHS,
    word_window = (0.1, 1.1),    # seconds into the word segment used for averaging
    gap_window  = (0.0, 0.5),    # seconds into the gap segment
)

grand = supersubject.build_grand_matrices(segs, df_features)   # .words_s0/.words_s1/.gaps_s0/.gaps_s1, .features, .word_labels
print(f'Grand words S0: {grand.words_s0.shape}  |  Grand gaps S0: {grand.gaps_s0.shape}')

## Step 6 - Population PSTH of the top-100 channels

**Why:** characterize the *population* response that the RSA actually runs on. The grand-average
PSTH of the top-100 channels (per session) shows the canonical word-by-word response across the
list, and confirms the selected channels carry a clean, time-locked signal. Computed both as
linear-mean -> dB and as the RAW (average-first-then-dB) version, matching the two PSTH styles
used in the analysis.

**Input:** `SEL` (full stitched traces) + `df_features` + `scores` (to rank the top-100).

**Output:** grand-average PSTH figures for ses-0 and ses-1 + an electrodes CSV of the selected set.

In [ ]:
viz.plot_population_psth(SEL, df_features, scores, top_n=100, session='ses-0', out=RESULTS / 'grand_psth_s0.png')
viz.plot_population_psth(SEL, df_features, scores, top_n=100, session='ses-1', out=RESULTS / 'grand_psth_s1.png')
viz.plot_population_psth(SEL, df_features, scores, top_n=100, session='ses-0', mode='raw_then_db',
                         out=RESULTS / 'grand_psth_raw_s0.png')

dataio.export_selected_electrodes(df_features, scores, top_n=100, out=RESULTS / 'top100_electrodes.csv')

## Step 7 - Word-identity RSM (300x300) + cross-session consistency

**Why:** beyond serial position, is the neural geometry of the **word identities themselves**
stable across two independent recording days? We build the full 300x300 word RSM per session
(Pearson similarity of every word pair over all channels) and test whether the S0 and S1 RSMs
agree (Pearson r + cosine over the upper triangle). High agreement => a reproducible word-level
representation, not measurement noise.

**Input:** `grand.words_s0/s1` (300 x N).

**Output:** the two 300x300 RSMs + a consistency scatter (S0 vs S1) + a 50-word zoom-in.

In [ ]:
rsm = rsa.word_rsm(grand.words_s0, grand.words_s1)   # all channels; .rsm_s0/.rsm_s1/.pearson_r/.cosine/.p_value
print(f'Word-RSM cross-session  Pearson r = {rsm.pearson_r:.3f}   cosine = {rsm.cosine:.3f}')

viz.plot_word_rsm(rsm, out=RESULTS / 'word_rsm.png')                                   # two 300x300 heatmaps
viz.plot_rsm_consistency_scatter(rsm, out=RESULTS / 'word_rsm_consistency_scatter.png')# S0 vs S1 pair similarity
viz.plot_word_rsm_zoom(rsm, n=50, labels=grand.word_labels, out=RESULTS / 'word_rsm_zoom.png')

## Step 8 - Serial-position RSA (within session) + decoding diagonal

**Why:** test whether each serial position (1-12) has a **distinct, reliable** neural
representation. We pick the 100 most informative channels by **positional variance on ses-0
only** (reused on ses-1, to avoid circularity), then use **split-half reliability** (x10):
each iteration splits the 25 lists into two independent halves and correlates position i in
half A with position j in half B. The **diagonal** of the 12x12 RDM is the decoding score per
position. We report the diagonal three ways: plain (mean of matrices), trial-wise (RDM-per-pair
then averaged), and with error bars from the 10 iterations.

**Input:** `grand.words_s0/s1` (reshaped 25x12xN); `n_channels=100`, `n_iter=10`.

**Output:** stable 12x12 RDMs per session (+ all iteration matrices), the locked channel
indices, word/gap RDM heatmaps, and the three decoding-diagonal bar plots.

In [ ]:
rdm_w_s0, iters_w_s0, idx_s0 = rsa.serial_position_rdm(grand.words_s0, n_channels=100, n_iter=10, select='variance')
# ses-1 reuses the ses-0 channels (selected on ses-0 only, to avoid circularity)
rdm_w_s1, iters_w_s1, idx_s1 = rsa.serial_position_rdm(grand.words_s1, fixed_indices=idx_s0, n_iter=10)

# Gaps reuse the SAME channels selected from the words analysis (fair cross-condition comparison)
rdm_g_s0, iters_g_s0, _ = rsa.serial_position_rdm(grand.gaps_s0, fixed_indices=idx_s0, n_iter=10)
rdm_g_s1, iters_g_s1, _ = rsa.serial_position_rdm(grand.gaps_s1, fixed_indices=idx_s0, n_iter=10)

viz.plot_rdm(rdm_w_s0, rdm_w_s1, title='Serial-position RDM (Words)', out=RESULTS / 'rdm_words.png')
viz.plot_rdm(rdm_g_s0, rdm_g_s1, title='Serial-position RDM (Gaps)',  out=RESULTS / 'rdm_gaps.png')

# Decoding diagonal - three variants
viz.plot_decoding_diagonal(rdm_w_s0, rdm_w_s1, out=RESULTS / 'decoding_diagonal.png')                        # plain (mean)
rdm_tw_s0 = rsa.trialwise_rdm(grand.words_s0, fixed_indices=idx_s0)
rdm_tw_s1 = rsa.trialwise_rdm(grand.words_s1, fixed_indices=idx_s0)
viz.plot_decoding_diagonal(rdm_tw_s0, rdm_tw_s1, out=RESULTS / 'decoding_diagonal_trialwise.png')            # trial-wise
viz.plot_decoding_diagonal_errorbars(iters_w_s0, iters_w_s1, out=RESULTS / 'decoding_diagonal_errorbars.png')# error bars

## Step 9 - Second-order RSA (relative representation + full matrices)

**Why:** is the representational **geometry** - which positions are similar to which -
reproduced across the two sessions? For each position i we correlate row i of the S0 RDM with
row i of the S1 RDM (the *relative representation* vector). Beyond the per-position vector, we
also compute the **full second-order matrix** (every S0 row vs every S1 row) for words and gaps,
whose diagonal is the relative representation and whose off-diagonal shows position confusions.

**Input:** the per-iteration RDM stacks for words and gaps.

**Output:** relative-representation bar plot (words & gaps, with SEM), the full second-order
contextual-similarity matrices, and their diagonals bar plot.

In [ ]:
relrep_words, relrep_words_sem = rsa.second_order_rsa(iters_w_s0, iters_w_s1)
relrep_gaps,  relrep_gaps_sem  = rsa.second_order_rsa(iters_g_s0, iters_g_s1)
viz.plot_relative_representation(relrep_words, relrep_words_sem,
                                 relrep_gaps,  relrep_gaps_sem,
                                 out=RESULTS / 'relative_representation.png')

# Full second-order matrices (every S0 row vs every S1 row), words & gaps
so_words = rsa.second_order_matrix(iters_w_s0, iters_w_s1)
so_gaps  = rsa.second_order_matrix(iters_g_s0, iters_g_s1)
viz.plot_second_order_matrices(so_words, so_gaps, out=RESULTS / 'second_order_full_matrices.png')
viz.plot_second_order_diagonals(so_words, so_gaps, out=RESULTS / 'second_order_diagonals_bars.png')

## Step 10 - Words vs Gaps (reinstatement)

**Why:** the headline question - does a word's neural code **persist into the silent gap**
after it disappears? Using split-half, we build a **cross-condition RDM**: cell [i, j] =
correlation of the *word* profile at position i with the *gap* profile at position j (both
split directions averaged). The **diagonal** [i, i] is reinstatement: the word-i code showing
up in gap i. We then test (a) whether that reinstatement profile is **consistent across
sessions**, and (b) the **second-order words x gaps geometry** (row-wise correlation of the
word RDM with the gap RDM).

**Input:** word & gap grand matrices on the shared selected channels; the word/gap RDMs.

**Output:** cross-condition RDM heatmap + diagonal bar plot, cross-session consistency, and the
words x gaps second-order matrix + its diagonal.

In [ ]:
cross_s0 = reinstatement.cross_condition_rdm(grand.words_s0, grand.gaps_s0, fixed_indices=idx_s0)
cross_s1 = reinstatement.cross_condition_rdm(grand.words_s1, grand.gaps_s1, fixed_indices=idx_s1)
viz.plot_cross_rdm([cross_s0, cross_s1], out=RESULTS / 'cross_rdm_heatmap.png')
viz.plot_cross_rdm_barplot([cross_s0, cross_s1], out=RESULTS / 'cross_rdm_barplot.png')

# (a) cross-session consistency of the reinstatement profile
consistency = reinstatement.cross_session_consistency(cross_s0, cross_s1)
viz.plot_cross_session_consistency(consistency, out=RESULTS / 'second_order_cross_rdm.png')

# (b) second-order Words x Gaps geometry
so_wg_s0 = rsa.words_gaps_second_order(rdm_w_s0, rdm_g_s0)
so_wg_s1 = rsa.words_gaps_second_order(rdm_w_s1, rdm_g_s1)
viz.plot_words_gaps_second_order([so_wg_s0, so_wg_s1], out=RESULTS / 'second_order_words_gaps.png')
viz.plot_words_gaps_second_order_diagonal([so_wg_s0, so_wg_s1], out=RESULTS / 'second_order_words_gaps_diagonal.png')

## Step 11 - Time-resolved decay + Maintenance

**Why:** two distinct questions about the gap.
- **Time-resolved (within a gap):** slice the gap into 50 ms windows and recompute the
  word-vs-gap second-order RSA per window - *when* does the trace decay right after the word?
  We save the **per-window gap RDMs**, the **per-window relative-representation bar plots**, a
  **summary heatmap** (position x time), and **decay trendlines** (primacy/middle/recency).
- **Maintenance (across the list):** correlate each position's word **template** with the
  neural profile of *every* gap in the list - *how far* does position i's code keep echoing.

**Input:** gap segments (50 ms windows) + per-position word templates, on the shared channels.

**Output:** per-window RDMs + per-window bars, summary heatmap, decay trend, per-position
maintenance bar plots.

In [ ]:
# Within-gap decay (per-window detail + summary)
tr = reinstatement.time_resolved_gap(SEL, PATHS, idx_s0, idx_s1, window_ms=50, n_windows=10)
viz.plot_time_resolved_rdms       (tr, out_dir=RESULTS / 'gap_rdm')                  # per-window gap RDM heatmaps
viz.plot_time_resolved_relrep_bars(tr, out_dir=RESULTS / 'relrep_per_window')        # per-window rel-rep bar plots
viz.plot_time_resolved_heatmap    (tr, out=RESULTS / 'time_resolved_heatmap.png')    # summary position x time
viz.plot_decay_trendlines         (tr, out=RESULTS / 'memory_decay_trend.png')       # primacy/middle/recency

# Across-list maintenance
maint = reinstatement.maintenance(grand.words_s0, grand.words_s1, SEL, PATHS, idx_s0, idx_s1,
                                  window_ms=50, n_windows=10)
viz.plot_maintenance(maint, out_dir=RESULTS / 'maintenance')

## Step 12 - Nearest-neighbor decoding

**Why:** a direct, classifier-free test - can we actually **name** the serial position behind
a neural pattern, discriminating against all 12 at once? Each position's S1 pattern is matched
to the most-correlated S0 template; correct = same position. Run in two spaces: the **RDM
space** (a position's similarity profile) and the **raw-activity space** (mean HFB over
channels). Chance = 1/12 ~ 8.3%.

**Input:** positional profiles - S0 as templates, S1 as test - for words and gaps.

**Output:** 12x12 confusion matrices + accuracy vs chance.

In [ ]:
acc_rdm = decoding.nearest_neighbor(rdm_w_s0, rdm_w_s1, space='rdm')
acc_raw = decoding.nearest_neighbor(grand.words_s0, grand.words_s1, space='raw', fixed_indices=idx_s0)

viz.plot_decoding_confusion(acc_rdm, acc_raw, chance=1/12, out=RESULTS / 'nn_decoding.png')
print(f'NN decoding accuracy  |  RDM space: {acc_rdm.accuracy:.3f}   raw space: {acc_raw.accuracy:.3f}   (chance 0.083)')

## Step 13 - Electrode anatomical brain map (interactive)

**Why:** put the analysis in anatomical context. This builds one **interactive HTML brain map
per subject** (all electrodes, colored by AAL region) plus a shared color legend, so coverage
can be inspected in 3D. The same pipeline assigns each contact its **AAL label** and **Broad_ROI**
(IFG / PFC / MTL / Temporal / Occipital / Other) - the anatomical tags that Step 4's region
filtering relies on. Built on `electrode_brain_map.py`.

**Input:** raw BIDS electrode `.tsv` (MNI152 coords) + the local AAL atlas (`ROI_MNI_V4.nii/.txt`).
Independent of the processed matrices, so it can run on its own.

**Output:** `brain_maps/{subject}_brain_map.html` (interactive) + `brain_maps/brain_plot_legend.png`;
updates each `{subject}_electrode_locations.csv` with `AAL_Label` + `Broad_ROI`.

In [ ]:
brainmap.generate_brain_maps(
    SUBJECTS, PATHS,
    out_dir = PATHS.results_dir / 'brain_maps',   # per-subject interactive HTML + shared legend PNG
    session = 'ses-0',                            # electrode coords are shared across sessions
)
# Side effect: writes/updates each {subject}_electrode_locations.csv with AAL_Label + Broad_ROI
# (the same anatomical labels Step 4 uses for region filtering).

## Outputs

All figures and tables are written under `results/reinstatement/`:

- **Behavioral (Step 1):** `SPC_curve.png`, `SPC_bars.png`, `FR1_behavioral_summary.csv`, `Subject_Level_SPC.csv`
- **PSTH (Steps 3, 6):** `psth_examples.png`, `grand_psth_s0.png`, `grand_psth_s1.png`, `grand_psth_raw_s0.png`, `top100_electrodes.csv`
- **Word RSM (Step 7):** `word_rsm.png`, `word_rsm_consistency_scatter.png`, `word_rsm_zoom.png`
- **Serial-position RSA (Step 8):** `rdm_words.png`, `rdm_gaps.png`, `decoding_diagonal.png`, `decoding_diagonal_trialwise.png`, `decoding_diagonal_errorbars.png`
- **Second-order RSA (Step 9):** `relative_representation.png`, `second_order_full_matrices.png`, `second_order_diagonals_bars.png`
- **Reinstatement / words vs gaps (Step 10):** `cross_rdm_heatmap.png`, `cross_rdm_barplot.png`, `second_order_cross_rdm.png`, `second_order_words_gaps.png`, `second_order_words_gaps_diagonal.png`
- **Time-resolved (Step 11):** `gap_rdm/*.png`, `relrep_per_window/*.png`, `time_resolved_heatmap.png`, `memory_decay_trend.png`, `maintenance/*.png`
- **Decoding (Step 12):** `nn_decoding.png`
- **Brain map (Step 13):** `brain_maps/{subject}_brain_map.html` (interactive), `brain_maps/brain_plot_legend.png`

**To re-run without preprocessing:** point `IEEG_DATA_ROOT` at the processed-data tier
(`dr-processed/`); the notebook runs end-to-end from there. *(Step 13 additionally reads the
raw BIDS electrode `.tsv` and the local AAL atlas; if those are absent it is skipped.)*

*Not included in this notebook (say the word to add):* SME / reinstatement-predicts-recall,
Context-Vector RDM (TCM, Manning 2011), per-word spaghetti plots.